# Dance Genre Classification — 4 Model Comparison
## AIST++ Dataset — 10 Genres

**Models compared:**

| Model | Type | Temporal | Spatial Graph |
|---|---|---|---|
| SVM (RBF) | Traditional ML | ❌ | ❌ |
| Random Forest | Traditional ML | ❌ | ❌ |
| LSTM | Deep Learning | ✅ | ❌ |
| ST-GCN | Deep Learning | ✅ | ✅ |

**Research question:** Does modeling temporal dynamics and spatial skeleton structure improve dance genre classification?

**References:**
- Tsuchida et al. (2019): AIST — proposed LSTM + SVM for dance genre classification
- Laban et al. (2025): RF 99.68%, SVM 99.07% with window=55 on AIST++
- Li et al. (2017): LSTM for skeleton action recognition
- Yan et al. (2018): ST-GCN — Spatial Temporal Graph Convolutional Networks

**Dataset:** AIST++ — 10 genres × 10 sequences (8 train + 2 test) = 100 sequences  
**Evaluation:** Sequence-level split (follows NTU RGB+D Cross-Subject protocol)

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {device.upper()}')


In [ ]:
!pip install scikit-learn matplotlib seaborn pandas numpy --quiet
print('All packages installed')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
 
import os
import pickle
import numpy as np
import pandas as pd
import time
 
PKL_DIR = '/content/drive/MyDrive/AIST_PKL'
AR_DIR  = '/content/drive/MyDrive/AIST_ActionRec_Final'
os.makedirs(AR_DIR, exist_ok=True)
 
GENRE_CODES = {
    'gBR': 'Break',
    'gPO': 'Pop',
    'gLO': 'Lock',
    'gWA': 'Waack',
    'gMH': 'Mid Hip-hop',
    'gLH': 'LA Hip-hop',
    'gHO': 'House',
    'gKR': 'Krump',
    'gJS': 'Street Jazz',
    'gJB': 'Ballet Jazz'
}
 
all_files = sorted(os.listdir(PKL_DIR))
print(f'Total .pkl files: {len(all_files)}')
print('Files per genre:')
for code, name in GENRE_CODES.items():
    n = len([f for f in all_files if f.startswith(code)])
    print(f'  {code} ({name:15s}): {n}')


In [ ]:
SEQUENCES = {}
for code, name in GENRE_CODES.items():
    genre_files = sorted([f for f in all_files if f.startswith(code)])
    SEQUENCES[name] = {
        'code':  code,
        'train': genre_files[:8],
        'test':  genre_files[-2:]
    }
 
GENRES       = list(SEQUENCES.keys())
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
NUM_CLASSES  = len(GENRES)
 
print(f'Genres ({NUM_CLASSES}): {GENRES}')
print('\nSplit:')
for name, s in SEQUENCES.items():
    print(f'  {name:15s}: {len(s["train"])} train | {len(s["test"])} test')


In [ ]:
COCO_EDGES = [
    (0, 1),  (0, 2),
    (1, 3),  (2, 4),
    (5, 6),
    (5, 7),  (7, 9),
    (6, 8),  (8, 10),
    (5, 11), (6, 12),
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
]
NUM_JOINTS = 17
 
 
def normalize_skeleton(keypoints):
    """
    Normalize skeleton by hip center subtraction and scaling.
    Following Two-Stream RNN (2017) and PYSKL (2022).

    Input:  (17, 3) — x, y, confidence
    Output: (34,)   — normalized x, y
    """
    xy = keypoints[:, :2].copy()
    hip_center = (xy[11] + xy[12]) / 2.0
    xy -= hip_center
    max_val = np.abs(xy).max()
    if max_val > 0:
        xy /= max_val
    return xy.flatten()
 
 
def load_pkl_sequence(pkl_path, camera_idx=0, frame_step=6):
    """
    Load keypoints from AIST++ .pkl file.
    Downsamples 60fps → 10fps.
    Returns list of (17, 2) normalized frames.
    """
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    kpts = data['keypoints2d'][camera_idx]
    frames = []
    for i in range(0, len(kpts), frame_step):
        norm_flat = normalize_skeleton(kpts[i])
        norm_2d   = norm_flat.reshape(17, 2)
        frames.append(norm_2d)
    return frames
 
 
def sliding_window_flat(frames, window_size=55, stride=10):
    """Sliding window → flattened (for SVM/RF): (N, window*34)"""
    windows = []
    for s in range(0, len(frames) - window_size + 1, stride):
        w = np.array(frames[s:s+window_size])
        windows.append(w.reshape(-1))
    return np.array(windows) if windows else np.empty((0, window_size*34))
 
 
def sliding_window_lstm(frames, window_size=55, stride=10):
    """Sliding window → (N, window, 34) for LSTM"""
    windows = []
    for s in range(0, len(frames) - window_size + 1, stride):
        w = np.array(frames[s:s+window_size])
        windows.append(w.reshape(window_size, 34))
    return np.array(windows) if windows else np.empty((0, window_size, 34))
 
 
def sliding_window_gcn(frames, window_size=55, stride=10):
    """Sliding window → (N, 2, window, 17) for ST-GCN"""
    windows = []
    for s in range(0, len(frames) - window_size + 1, stride):
        w = np.array(frames[s:s+window_size])
        w = w.transpose(2, 0, 1)
        windows.append(w)
    return np.array(windows) if windows else np.empty((0, 2, window_size, 17))
 
 
test_file   = f"{PKL_DIR}/{SEQUENCES['Break']['train'][0]}"
test_frames = load_pkl_sequence(test_file)
print(f'Frames (60→10fps): {len(test_frames)}')
print(f'Flat window:  {sliding_window_flat(test_frames).shape}')
print(f'LSTM window:  {sliding_window_lstm(test_frames).shape}')
print(f'GCN window:   {sliding_window_gcn(test_frames).shape}')


In [ ]:
WINDOW_SIZE = 55
STRIDE      = 10
 
X_flat_train, X_lstm_train, X_gcn_train, y_train = [], [], [], []
X_flat_test,  X_lstm_test,  X_gcn_test,  y_test  = [], [], [], []
 
print('Loading 10 genres...')
print('=' * 65)
 
for genre, splits in SEQUENCES.items():
    label = GENRE_TO_IDX[genre]
    tr_win = te_win = 0
 
    for pkl_file in splits['train']:
        path = f'{PKL_DIR}/{pkl_file}'
        if not os.path.exists(path): continue
        frames = load_pkl_sequence(path)
        wf = sliding_window_flat(frames, WINDOW_SIZE, STRIDE)
        wl = sliding_window_lstm(frames, WINDOW_SIZE, STRIDE)
        wg = sliding_window_gcn(frames,  WINDOW_SIZE, STRIDE)
        if len(wf) > 0:
            X_flat_train.extend(wf)
            X_lstm_train.extend(wl)
            X_gcn_train.extend(wg)
            y_train.extend([label] * len(wf))
            tr_win += len(wf)
 
    for pkl_file in splits['test']:
        path = f'{PKL_DIR}/{pkl_file}'
        if not os.path.exists(path): continue
        frames = load_pkl_sequence(path)
        wf = sliding_window_flat(frames, WINDOW_SIZE, STRIDE)
        wl = sliding_window_lstm(frames, WINDOW_SIZE, STRIDE)
        wg = sliding_window_gcn(frames,  WINDOW_SIZE, STRIDE)
        if len(wf) > 0:
            X_flat_test.extend(wf)
            X_lstm_test.extend(wl)
            X_gcn_test.extend(wg)
            y_test.extend([label] * len(wf))
            te_win += len(wf)
 
    print(f'{genre:15s}: train={tr_win:3d} | test={te_win:3d} windows')
 
X_flat_train = np.array(X_flat_train)
X_lstm_train = np.array(X_lstm_train)
X_gcn_train  = np.array(X_gcn_train)
y_train      = np.array(y_train)
 
X_flat_test  = np.array(X_flat_test)
X_lstm_test  = np.array(X_lstm_test)
X_gcn_test   = np.array(X_gcn_test)
y_test       = np.array(y_test)
 
print('=' * 65)
print(f'SVM/RF input:  {X_flat_train.shape} train | {X_flat_test.shape} test')
print(f'LSTM input:    {X_lstm_train.shape} train | {X_lstm_test.shape} test')
print(f'ST-GCN input:  {X_gcn_train.shape} train | {X_gcn_test.shape} test')


In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
 
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_flat_train)
X_te_scaled = scaler.transform(X_flat_test)
 
print('Training SVM...')
t0 = time.time()
svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm.fit(X_tr_scaled, y_train)
svm_time = time.time() - t0
svm_preds = svm.predict(X_te_scaled)
svm_acc   = accuracy_score(y_test, svm_preds) * 100
svm_f1    = f1_score(y_test, svm_preds, average='macro') * 100
print(f'SVM: Accuracy={svm_acc:.2f}% | F1={svm_f1:.2f}% | Time={svm_time:.1f}s')
 
print('\nTraining Random Forest...')
t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_flat_train, y_train)
rf_time = time.time() - t0
rf_preds = rf.predict(X_flat_test)
rf_acc   = accuracy_score(y_test, rf_preds) * 100
rf_f1    = f1_score(y_test, rf_preds, average='macro') * 100
print(f'RF:  Accuracy={rf_acc:.2f}% | F1={rf_f1:.2f}% | Time={rf_time:.1f}s')


In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
 
class SkeletonDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]
 
 
class DanceGenreLSTM(nn.Module):
    """
    2-layer LSTM for dance genre classification.
    Following Li et al. (2017) and Tsuchida et al. (2019).

    Reads skeleton frames sequentially (t=0..54).
    Uses last hidden state for classification.
    No spatial graph — treats joints as flat feature vector.
    """
    def __init__(self, input_size=34, hidden_size=256,
                 num_layers=2, num_classes=10, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, num_classes)
 
    def forward(self, x):
        
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(self.dropout(out))
 
 
def train_model(model, train_loader, test_loader, num_epochs=150, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = StepLR(optimizer, step_size=50, gamma=0.5)
 
    train_losses, train_accs, test_accs = [], [], []
    best_acc, best_epoch = 0.0, 0
    model_name = model.__class__.__name__
 
    for epoch in range(num_epochs):
        model.train()
        loss_sum = correct = total = 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            out  = model(X_b)
            loss = criterion(out, y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            loss_sum += loss.item()
            correct  += (out.argmax(1) == y_b).sum().item()
            total    += y_b.size(0)
        train_acc = 100 * correct / total
 
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for X_b, y_b in test_loader:
                out = model(X_b.to(device))
                correct += (out.argmax(1) == y_b.to(device)).sum().item()
                total   += y_b.size(0)
        test_acc = 100 * correct / total
 
        scheduler.step()
        train_losses.append(loss_sum / len(train_loader))
        train_accs.append(train_acc)
        test_accs.append(test_acc)
 
        if test_acc > best_acc:
            best_acc   = test_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), f'best_{model_name}.pt')
 
        if (epoch + 1) % 30 == 0:
            print(f'[{model_name}] Epoch {epoch+1:3d}/{num_epochs} | '
                  f'Loss: {loss_sum/len(train_loader):.4f} | '
                  f'Train: {train_acc:.1f}% | Test: {test_acc:.1f}%')
 
    return best_acc, best_epoch, train_losses, train_accs, test_accs
 
 
BATCH_SIZE = 64
lstm_train_loader = DataLoader(SkeletonDataset(X_lstm_train, y_train),
                               batch_size=BATCH_SIZE, shuffle=True)
lstm_test_loader  = DataLoader(SkeletonDataset(X_lstm_test,  y_test),
                               batch_size=BATCH_SIZE, shuffle=False)
 
lstm_model = DanceGenreLSTM(
    input_size=34, hidden_size=256,
    num_layers=2, num_classes=10, dropout=0.3
).to(device)
 
print(f'LSTM parameters: {sum(p.numel() for p in lstm_model.parameters()):,}')
print('Training LSTM...')
print('=' * 65)
t0 = time.time()
lstm_best_acc, lstm_best_epoch, lstm_losses, lstm_train_accs, lstm_test_accs = \
    train_model(lstm_model, lstm_train_loader, lstm_test_loader)
lstm_time = time.time() - t0
print('=' * 65)
print(f'LSTM Best: {lstm_best_acc:.2f}% at epoch {lstm_best_epoch} | Time: {lstm_time:.0f}s')


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
 
 
def build_adjacency_matrix(num_joints=17, edges=None):
    """
    Build normalized adjacency matrix for COCO 17 keypoints.
    Following Yan et al. (2018) ST-GCN graph construction.

    Returns: (17, 17) normalized adjacency matrix
    """
    if edges is None:
        edges = COCO_EDGES
 
    A = np.zeros((num_joints, num_joints))
 
    
    for i in range(num_joints):
        A[i, i] = 1
 
    
    for i, j in edges:
        A[i, j] = 1
        A[j, i] = 1
 
    
    D = np.diag(A.sum(axis=1) ** -0.5)
    A_norm = D @ A @ D
 
    return torch.FloatTensor(A_norm)
 
 
class GraphConv(nn.Module):
    """
    Spatial Graph Convolution.
    Propagates features along skeleton graph edges.
    Each joint aggregates information from connected joints.

    Input:  (batch, C_in, T, V)
    Output: (batch, C_out, T, V)
    """
    def __init__(self, in_channels, out_channels, A):
        super().__init__()
        self.A    = A
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
 
    def forward(self, x):
        
        A = self.A.to(x.device)
        
        x = torch.einsum('nctv,vw->nctw', x, A)
        return self.conv(x)
 
 
class STGCNBlock(nn.Module):
    """
    ST-GCN Block: Spatial Graph Conv + Temporal Conv.
    Following Yan et al. (2018).

    Spatial:  aggregates features from connected joints
    Temporal: applies 1D convolution along time axis
    """
    def __init__(self, in_ch, out_ch, A, stride=1, dropout=0.3):
        super().__init__()
        self.gcn = GraphConv(in_ch, out_ch, A)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            
            nn.Conv2d(out_ch, out_ch,
                      kernel_size=(9, 1),
                      padding=(4, 0),
                      stride=(stride, 1)),
            nn.BatchNorm2d(out_ch),
            nn.Dropout(dropout)
        )
        
        self.residual = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=(stride, 1)),
            nn.BatchNorm2d(out_ch)
        ) if in_ch != out_ch or stride != 1 else nn.Identity()
        self.relu = nn.ReLU()
 
    def forward(self, x):
        res = self.residual(x)
        x   = self.tcn(self.gcn(x))
        return self.relu(x + res)
 
 
class STGCN(nn.Module):
    """
    Spatial Temporal Graph Convolutional Network.
    Yan et al. (2018), adapted for COCO 17 keypoints.

    Architecture: 6 ST-GCN blocks with increasing channels.
    Smaller than original (10 blocks) due to smaller dataset.

    Input:  (batch, C=2, T=55, V=17)
    Output: (batch, num_classes)
    """
    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()
        A = build_adjacency_matrix()
 
        self.data_bn = nn.BatchNorm1d(2 * NUM_JOINTS)
 
        self.blocks = nn.ModuleList([
            STGCNBlock(2,   64,  A, dropout=dropout),
            STGCNBlock(64,  64,  A, dropout=dropout),
            STGCNBlock(64,  128, A, stride=2, dropout=dropout),
            STGCNBlock(128, 128, A, dropout=dropout),
            STGCNBlock(128, 256, A, stride=2, dropout=dropout),
            STGCNBlock(256, 256, A, dropout=dropout),
        ])
 
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc   = nn.Linear(256, num_classes)
 
    def forward(self, x):
        
        N, C, T, V = x.shape
 
        
        x = x.permute(0, 3, 1, 2).reshape(N, V*C, T)
        x = self.data_bn(x)
        x = x.reshape(N, V, C, T).permute(0, 2, 3, 1)
 
        
        for block in self.blocks:
            x = block(x)
 
        
        x = self.pool(x).reshape(N, -1)
        return self.fc(x)
 
 
A = build_adjacency_matrix()
print('Adjacency matrix shape:', A.shape)
print('Non-zero connections:', (A > 0).sum().item())
 
gcn_train_loader = DataLoader(SkeletonDataset(X_gcn_train, y_train),
                              batch_size=BATCH_SIZE, shuffle=True)
gcn_test_loader  = DataLoader(SkeletonDataset(X_gcn_test,  y_test),
                              batch_size=BATCH_SIZE, shuffle=False)
 
stgcn_model = STGCN(num_classes=10, dropout=0.3).to(device)
print(f'ST-GCN parameters: {sum(p.numel() for p in stgcn_model.parameters()):,}')
print('Training ST-GCN...')
print('=' * 65)
t0 = time.time()
gcn_best_acc, gcn_best_epoch, gcn_losses, gcn_train_accs, gcn_test_accs = \
    train_model(stgcn_model, gcn_train_loader, gcn_test_loader)
gcn_time = time.time() - t0
print('=' * 65)
print(f'ST-GCN Best: {gcn_best_acc:.2f}% at epoch {gcn_best_epoch} | Time: {gcn_time:.0f}s')


In [ ]:
from sklearn.metrics import confusion_matrix
 
def get_predictions(model, loader, model_path):
    model.load_state_dict(torch.load(model_path))
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            out = model(X_b.to(device))
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(y_b.numpy())
    return np.array(preds), np.array(labels)
 
lstm_preds,  lstm_labels  = get_predictions(lstm_model,  lstm_test_loader,  'best_DanceGenreLSTM.pt')
stgcn_preds, stgcn_labels = get_predictions(stgcn_model, gcn_test_loader,   'best_STGCN.pt')
 
lstm_acc  = accuracy_score(lstm_labels,  lstm_preds)  * 100
lstm_f1   = f1_score(lstm_labels,  lstm_preds,  average='macro') * 100
stgcn_acc = accuracy_score(stgcn_labels, stgcn_preds) * 100
stgcn_f1  = f1_score(stgcn_labels, stgcn_preds, average='macro') * 100
 
results = pd.DataFrame([
    {'Model': 'SVM (RBF)',     'Type': 'Traditional ML', 'Temporal': 'No',  'Spatial Graph': 'No',
     'Accuracy': round(svm_acc, 2),   'Macro F1': round(svm_f1, 2),   'Train Time(s)': round(svm_time, 1)},
    {'Model': 'Random Forest', 'Type': 'Traditional ML', 'Temporal': 'No',  'Spatial Graph': 'No',
     'Accuracy': round(rf_acc, 2),    'Macro F1': round(rf_f1, 2),    'Train Time(s)': round(rf_time, 1)},
    {'Model': 'LSTM',          'Type': 'Deep Learning',  'Temporal': 'Yes', 'Spatial Graph': 'No',
     'Accuracy': round(lstm_acc, 2),  'Macro F1': round(lstm_f1, 2),  'Train Time(s)': round(lstm_time, 1)},
    {'Model': 'ST-GCN',        'Type': 'Deep Learning',  'Temporal': 'Yes', 'Spatial Graph': 'Yes',
     'Accuracy': round(stgcn_acc, 2), 'Macro F1': round(stgcn_f1, 2), 'Train Time(s)': round(gcn_time, 1)},
])
 
print('=' * 70)
print('FINAL COMPARISON — 4 Models on AIST++ (10 Dance Genres)')
print('=' * 70)
print(results.to_string(index=False))
results.to_csv('final_comparison.csv', index=False)
 
genre_rows = []
for idx, genre in enumerate(GENRES):
    genre_rows.append({
        'Genre':      genre,
        'SVM F1':     round(f1_score(y_test==idx, svm_preds==idx,   zero_division=0)*100, 2),
        'RF F1':      round(f1_score(y_test==idx, rf_preds==idx,    zero_division=0)*100, 2),
        'LSTM F1':    round(f1_score(lstm_labels==idx,  lstm_preds==idx,  zero_division=0)*100, 2),
        'ST-GCN F1':  round(f1_score(stgcn_labels==idx, stgcn_preds==idx, zero_division=0)*100, 2),
    })
genre_df = pd.DataFrame(genre_rows)
print('\n=== F1 PER GENRE ===')
print(genre_df.to_string(index=False))
genre_df.to_csv('genre_f1_final.csv', index=False)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
 
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
model_names = ['SVM', 'Random\nForest', 'LSTM', 'ST-GCN']
 
accs = [svm_acc, rf_acc, lstm_acc, stgcn_acc]
bars = axes[0,0].bar(model_names, accs, color=colors, alpha=0.9)
for bar, v in zip(bars, accs):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, v+0.3,
                   f'{v:.1f}%', ha='center', fontweight='bold', fontsize=10)
axes[0,0].set_ylabel('Accuracy (%)')
axes[0,0].set_title('Overall Accuracy')
axes[0,0].set_ylim(0, 115)
axes[0,0].grid(axis='y', alpha=0.3)
 
f1s = [svm_f1, rf_f1, lstm_f1, stgcn_f1]
bars = axes[0,1].bar(model_names, f1s, color=colors, alpha=0.9)
for bar, v in zip(bars, f1s):
    axes[0,1].text(bar.get_x()+bar.get_width()/2, v+0.3,
                   f'{v:.1f}%', ha='center', fontweight='bold', fontsize=10)
axes[0,1].set_ylabel('Macro F1 (%)')
axes[0,1].set_title('Macro F1 Score')
axes[0,1].set_ylim(0, 115)
axes[0,1].grid(axis='y', alpha=0.3)
 
x = range(len(GENRES))
w = 0.2
for i, (col, name, vals) in enumerate(zip(
    colors, ['SVM', 'RF', 'LSTM', 'ST-GCN'],
    [genre_df['SVM F1'], genre_df['RF F1'],
     genre_df['LSTM F1'], genre_df['ST-GCN F1']]
)):
    axes[0,2].bar([j + i*w - 1.5*w for j in x], vals, w,
                  label=name, color=col, alpha=0.9)
axes[0,2].set_xticks(list(x))
axes[0,2].set_xticklabels(GENRES, rotation=20, ha='right', fontsize=7)
axes[0,2].set_ylabel('F1 (%)')
axes[0,2].set_title('F1 per Genre')
axes[0,2].legend(fontsize=8)
axes[0,2].set_ylim(0, 120)
axes[0,2].grid(axis='y', alpha=0.3)
 
axes[1,0].plot(lstm_train_accs, color='#4CAF50', label='Train')
axes[1,0].plot(lstm_test_accs,  color='#FF9800', label='Test')
axes[1,0].axvline(x=lstm_best_epoch-1, color='red', linestyle='--',
                  label=f'Best ({lstm_best_epoch})')
axes[1,0].set_xlabel('Epoch')
axes[1,0].set_ylabel('Accuracy (%)')
axes[1,0].set_title('LSTM Training Curves')
axes[1,0].legend()
axes[1,0].grid(alpha=0.3)
 
axes[1,1].plot(gcn_train_accs, color='#4CAF50', label='Train')
axes[1,1].plot(gcn_test_accs,  color='#E91E63', label='Test')
axes[1,1].axvline(x=gcn_best_epoch-1, color='red', linestyle='--',
                  label=f'Best ({gcn_best_epoch})')
axes[1,1].set_xlabel('Epoch')
axes[1,1].set_ylabel('Accuracy (%)')
axes[1,1].set_title('ST-GCN Training Curves')
axes[1,1].legend()
axes[1,1].grid(alpha=0.3)
 
cm = confusion_matrix(stgcn_labels, stgcn_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
short_g = [g[:7] for g in GENRES]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=short_g, yticklabels=short_g, ax=axes[1,2])
axes[1,2].set_xlabel('Predicted')
axes[1,2].set_ylabel('True')
axes[1,2].set_title('ST-GCN Confusion Matrix')
axes[1,2].tick_params(axis='x', rotation=30)
 
plt.suptitle(
    f'Dance Genre Classification — AIST++ (10 genres)\n'
    f'SVM: {svm_acc:.1f}% | RF: {rf_acc:.1f}% | '
    f'LSTM: {lstm_acc:.1f}% | ST-GCN: {stgcn_acc:.1f}%',
    fontsize=13
)
plt.tight_layout()
plt.savefig('final_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Charts saved')


In [ ]:
import json, shutil
 
pd.DataFrame({
    'epoch': range(1, 151),
    'lstm_train': lstm_train_accs, 'lstm_test': lstm_test_accs,
    'stgcn_train': gcn_train_accs, 'stgcn_test': gcn_test_accs
}).to_csv('training_history.csv', index=False)
 
summary = {
    'dataset':   'AIST++ — 10 dance genres',
    'protocol':  'Sequence-level split (NTU RGB+D Cross-Subject)',
    'window':    f'{WINDOW_SIZE} frames = {WINDOW_SIZE/10:.1f}s at 10fps',
    'train':     int(len(X_flat_train)),
    'test':      int(len(X_flat_test)),
    'SVM':       {'accuracy': round(svm_acc, 2),   'f1': round(svm_f1, 2)},
    'RF':        {'accuracy': round(rf_acc, 2),    'f1': round(rf_f1, 2)},
    'LSTM':      {'accuracy': round(lstm_acc, 2),  'f1': round(lstm_f1, 2),
                  'best_epoch': lstm_best_epoch},
    'ST-GCN':    {'accuracy': round(stgcn_acc, 2), 'f1': round(stgcn_f1, 2),
                  'best_epoch': gcn_best_epoch},
}
with open('summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
 
files = [
    'best_DanceGenreLSTM.pt', 'best_STGCN.pt',
    'final_results.png', 'final_comparison.csv',
    'genre_f1_final.csv', 'training_history.csv', 'summary.json'
]
for f in files:
    shutil.copy(f, f'{AR_DIR}/{f}')
    print(f'Saved: {f}')
 
print(f'\nAll results → AIST_ActionRec_Final/')
print()
print('=' * 60)
print('SUMMARY')
print('=' * 60)
print(results[['Model','Accuracy','Macro F1']].to_string(index=False))
print()
winner = results.loc[results['Accuracy'].idxmax(), 'Model']
print(f'Best model: {winner} ({results["Accuracy"].max():.2f}%)')
